Đọc dữ liệu

In [2]:

import os
import numpy as np
import pandas as pd

# Đường dẫn dữ liệu giống Decision Tree
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc dữ liệu
diamonds = pd.read_csv(file_path, sep=',')

# In kích thước dữ liệu
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

print("\n--- 5 dòng đầu tiên ---")
print(diamonds.head())

print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
print(f"Cut: {diamonds['cut'].unique()}")
print(f"Color: {diamonds['color'].unique()}")
print(f"Clarity: {diamonds['clarity'].unique()}")

print("\n--- Thống kê mô tả ---")
print(diamonds.describe())


Kích thước tập dữ liệu ban đầu: (53943, 11)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  object 
 3   color       53943 non-null  object 
 4   clarity     53943 non-null  object 
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB

--- 5 dòng đầu tiên ---
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  

Tiền xử lý dữ liệu

In [3]:

from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
if 'Unnamed: 0' in diamonds.columns:
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (x, y, z <= 0)
initial_rows = len(diamonds)

diamonds = diamonds[
    (diamonds['x'] > 0) &
    (diamonds['y'] > 0) &
    (diamonds['z'] > 0)
].copy()

invalid_dimensions = initial_rows - len(diamonds)

print(
    f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu "
    "(x, y, z <= 0)."
)

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
num_duplicates = diamonds.duplicated().sum()

diamonds = diamonds.drop_duplicates()

print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

clarity_categories = [
    'I1', 'SI2', 'SI1', 'VS2',
    'VS1', 'VVS2', 'VVS1', 'IF'
]

encoder = OrdinalEncoder(
    categories=[
        cut_categories,
        color_categories,
        clarity_categories
    ]
)

diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(
    diamonds[['cut', 'color', 'clarity']]
)

# 5: Kiểm tra dữ liệu sau tiền xử lý
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")
print(diamonds.head())



Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
0   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
1   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
2   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
3   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
4   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia tập dữ liệu

In [4]:

from sklearn.model_selection import KFold

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])

# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu thành 5 fold
Kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


Training và tìm siêu tham số bằng K-Fold

In [5]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import numpy as np
import pandas as pd
import time

# Các giá trị tham số cần thử (giống mức 3 Hold-out)
n_estimators_values = [50, 100]
max_depth_values = [10, 15]
min_samples_leaf_values = [1, 2]
min_samples_split_values = [2, 5]

# Lưu kết quả của từng Fold
results = []
best_params_per_fold = []

# Bắt đầu đo tổng thời gian
start_time = time.perf_counter()

# Duyệt qua 5 Fold
for fold, (train_index, test_index) in enumerate(Kf.split(X), start=1):

    print(f"\n========== FOLD {fold}/5 ==========")

    # Chia dữ liệu theo Fold
    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    # Tách Validation từ Train của Fold
    X_train_sub, X_val, y_train_sub, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        random_state=42
    )

    # Khởi tạo biến lưu bộ tham số tốt nhất
    best_rmse = float('inf')
    best_params = None
    count = 0

    # Thử 16 tổ hợp tham số
    for n_estimators in n_estimators_values:
        for max_depth in max_depth_values:
            for min_samples_leaf in min_samples_leaf_values:
                for min_samples_split in min_samples_split_values:

                    count += 1

                    model = RandomForestRegressor(
                        criterion="squared_error",
                        random_state=42,
                        n_estimators=n_estimators,
                        max_depth=max_depth,
                        min_samples_leaf=min_samples_leaf,
                        min_samples_split=min_samples_split
                    )

                    # Huấn luyện trên Train phụ
                    model.fit(X_train_sub, y_train_sub)

                    # Dự đoán trên Validation
                    y_pred_val = model.predict(X_val)

                    # Tính RMSE Validation
                    rmse = np.sqrt(
                        mean_squared_error(y_val, y_pred_val)
                    )

                    # Cập nhật bộ tham số tốt nhất
                    if rmse < best_rmse:
                        best_rmse = rmse
                        best_params = {
                            "n_estimators": n_estimators,
                            "max_depth": max_depth,
                            "min_samples_leaf": min_samples_leaf,
                            "min_samples_split": min_samples_split
                        }

    print(f"Đã hoàn thành tìm kiếm {count}/16 tổ hợp")
    print(f"Tham số tốt nhất Fold {fold}: {best_params}")
    print(f"RMSE Validation tốt nhất: {best_rmse:.4f}")

    # Lưu tham số tốt nhất của Fold
    best_params_per_fold.append({
        "Fold": fold,
        **best_params,
        "RMSE Validation": best_rmse
    })

    # Huấn luyện lại trên toàn bộ Train của Fold
    best_model = RandomForestRegressor(
        criterion="squared_error",
        random_state=42,
        **best_params
    )

    start_train_time = time.perf_counter()

    best_model.fit(X_train, y_train)

    training_time = time.perf_counter() - start_train_time

    # Dự đoán trên Train và Test của Fold
    y_pred_train = best_model.predict(X_train)
    y_pred_test = best_model.predict(X_test)

    # Tính các chỉ số Train
    mae_train = mean_absolute_error(y_train, y_pred_train)
    mse_train = mean_squared_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mse_train)
    r2_train = r2_score(y_train, y_pred_train)

    # Tính các chỉ số Test
    mae_test = mean_absolute_error(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mse_test)
    r2_test = r2_score(y_test, y_pred_test)

    # Lưu kết quả của Fold
    results.append({
        "Fold": fold,
        "MAE Train": mae_train,
        "MSE Train": mse_train,
        "RMSE Train": rmse_train,
        "R2 Train": r2_train,
        "MAE Test": mae_test,
        "MSE Test": mse_test,
        "RMSE Test": rmse_test,
        "R2 Test": r2_test,
        "Training Time (s)": training_time
    })

    print(f"RMSE Test Fold {fold}: {rmse_test:.4f}")
    print(f"Đã hoàn thành Fold {fold}/5")

# Tính tổng thời gian
train_time = time.perf_counter() - start_time

# Tạo bảng kết quả từng Fold
results_df = pd.DataFrame(results)

# Tính trung bình 5 Fold
mean_result = results_df.drop(columns=["Fold"]).mean()

mean_row = pd.DataFrame([mean_result])
mean_row.insert(0, "Fold", "Average")

# Ghép bảng kết quả
final_results = pd.concat(
    [results_df, mean_row],
    ignore_index=True
)

# Bảng tham số tốt nhất từng Fold
best_params_df = pd.DataFrame(best_params_per_fold)

print("\nKẾT QUẢ RANDOM FOREST - K-FOLD MỨC 3")
print(final_results.to_string(index=False, float_format="%.4f"))

print("\nTHAM SỐ TỐT NHẤT CỦA TỪNG FOLD")
print(best_params_df.to_string(index=False, float_format="%.4f"))

print(f"\nTổng thời gian thực hiện: {train_time:.4f} giây")



========== FOLD 1/5 ==========
Đã hoàn thành tìm kiếm 16/16 tổ hợp
Tham số tốt nhất Fold 1: {'n_estimators': 100, 'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 5}
RMSE Validation tốt nhất: 551.4814
RMSE Test Fold 1: 523.2357
Đã hoàn thành Fold 1/5

========== FOLD 2/5 ==========
Đã hoàn thành tìm kiếm 16/16 tổ hợp
Tham số tốt nhất Fold 2: {'n_estimators': 100, 'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 5}
RMSE Validation tốt nhất: 523.5830
RMSE Test Fold 2: 530.8491
Đã hoàn thành Fold 2/5

========== FOLD 3/5 ==========
Đã hoàn thành tìm kiếm 16/16 tổ hợp
Tham số tốt nhất Fold 3: {'n_estimators': 50, 'max_depth': 15, 'min_samples_leaf': 1, 'min_samples_split': 2}
RMSE Validation tốt nhất: 522.4679
RMSE Test Fold 3: 535.3977
Đã hoàn thành Fold 3/5

========== FOLD 4/5 ==========
Đã hoàn thành tìm kiếm 16/16 tổ hợp
Tham số tốt nhất Fold 4: {'n_estimators': 100, 'max_depth': 15, 'min_samples_leaf': 2, 'min_samples_split': 5}
RMSE Validation tốt nhất: 533.

In [6]:

# In bảng kết quả rút gọn
print("KẾT QUẢ RANDOM FOREST K-FOLD MỨC 3")

print(
    final_results[
        ["Fold", "RMSE Train", "RMSE Test", "R2 Test"]
    ].to_string(index=False)
)

print("\nTHAM SỐ TỐT NHẤT TỪNG FOLD")
print(
    best_params_df.to_string(index=False)
)

print(f"\nTổng thời gian: {train_time:.4f} giây")


KẾT QUẢ RANDOM FOREST K-FOLD MỨC 3
   Fold  RMSE Train  RMSE Test  R2 Test
      1  329.577883 523.235660 0.982676
      2  325.450495 530.849124 0.981893
      3  288.586167 535.397672 0.981922
      4  329.900190 534.189138 0.982438
      5  322.330138 567.236542 0.979838
Average  319.168974 538.181627 0.981753

THAM SỐ TỐT NHẤT TỪNG FOLD
 Fold  n_estimators  max_depth  min_samples_leaf  min_samples_split  RMSE Validation
    1           100         15                 2                  5       551.481438
    2           100         15                 2                  5       523.582970
    3            50         15                 1                  2       522.467880
    4           100         15                 2                  5       533.529200
    5           100         15                 2                  5       528.587686

Tổng thời gian: 709.4326 giây
